# EA3 - Actividad 3.2: Spark Structured Streaming

## Objetivos
- Leer streams de datos desde Kafka con `readStream`
- Parsear mensajes JSON desde Kafka
- Aplicar window aggregations con watermarks
- Escribir resultados con `writeStream` a diferentes destinos

> **IMPORTANTE:** Este notebook requiere el perfil **completo** de Docker.
> ```bash
> docker-compose --profile completo up -d
> ```
> Espera ~1 minuto para que Kafka y los demas servicios inicien completamente.

## Conceptos Clave

### Structured Streaming

**Spark Structured Streaming** es el motor de procesamiento de streams de Spark. Trata un stream de datos como una **tabla ilimitada** que crece continuamente a medida que llegan nuevos datos.

```
  Stream de entrada          "Tabla ilimitada"            Resultados
  =================          =================            ==========

  Datos en                   +---+---+---+---+
  tiempo t=1  ------>        | a | b | c | d |  ------>   Resultado t=1
                             +---+---+---+---+
  Datos en                   | e | f | g | h |
  tiempo t=2  ------>        +---+---+---+---+  ------>   Resultado t=2
                             | i | j | k | l |
  Datos en                   +---+---+---+---+
  tiempo t=3  ------>        | m | n | o | p |  ------>   Resultado t=3
                             +---+---+---+---+
```

### Micro-Batches

Spark procesa el stream en **micro-batches**: acumula datos durante un intervalo de tiempo (trigger) y los procesa como un mini-batch. Esto combina la simplicidad del batch con la inmediatez del streaming.

### Watermarks

Un **watermark** define cuanto tiempo Spark espera datos retrasados. Si un evento llega despues del watermark, se descarta. Esto permite manejar datos que llegan fuera de orden.

```
  Tiempo real:     t=0     t=10    t=20    t=30    t=40
                    |       |       |       |       |
  Watermark (30s):  |       |       |  <--- descarta datos anteriores a t=0
```

### Window Functions

Las **funciones de ventana** agrupan eventos por rangos de tiempo. Ejemplo: "total de ventas cada 1 minuto".

```
  |--- Ventana 1 ---|--- Ventana 2 ---|--- Ventana 3 ---|
  |  12:00 - 12:01  |  12:01 - 12:02  |  12:02 - 12:03  |
  |  [e1] [e2] [e3] |  [e4] [e5]      |  [e6] [e7] [e8] |
  |  sum = 15000     |  sum = 8500     |  sum = 22000    |
```

### Output Modes

| Modo | Descripcion | Uso tipico |
|------|-------------|------------|
| **append** | Solo escribe filas nuevas | Escritura a archivos, tablas |
| **complete** | Reescribe toda la tabla de resultados | Agregaciones (count, sum) |
| **update** | Solo escribe filas que cambiaron | Actualizaciones incrementales |

## Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("structured_streaming") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("Setup completado con soporte Kafka.")

## 1. Generar Datos en Kafka

Antes de leer un stream, necesitamos que haya datos fluyendo. Iniciamos el generador de datos en segundo plano para que envie transacciones a Kafka.

In [ ]:
import subprocess

# Iniciar generador en segundo plano: 5 eventos/seg por 2 minutos
proceso_generador = subprocess.Popen([
    "python", "/home/jovyan/scripts/generar_datos_streaming.py",
    "--tipo", "transacciones", "--velocidad", "5", "--duracion", "120",
    "--topic", "bigdata-transacciones"
])
print("Generador iniciado en segundo plano (5 eventos/seg por 2 min)")
print("Espera unos segundos antes de ejecutar la siguiente celda...")

## 2. Leer Stream desde Kafka

Usamos `readStream` para crear un DataFrame de streaming que lee continuamente de un topic de Kafka. Los mensajes de Kafka tienen la estructura: key, value, topic, partition, offset, timestamp.

In [ ]:
df_raw = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "bigdata-transacciones") \
    .option("startingOffsets", "earliest") \
    .load()

# Ver el esquema del stream de Kafka (key, value, topic, partition, offset, timestamp)
df_raw.printSchema()

## 3. Parsear JSON desde el Campo `value`

El contenido del mensaje esta en el campo `value` como bytes. Necesitamos:
1. Convertirlo a string
2. Parsear el JSON segun un esquema definido

In [ ]:
# Definir el esquema de los mensajes JSON
schema = StructType([
    StructField("id", StringType()),
    StructField("timestamp", StringType()),
    StructField("monto", IntegerType()),
    StructField("tienda_id", IntegerType()),
    StructField("producto", StringType()),
    StructField("cantidad", IntegerType()),
    StructField("metodo_pago", StringType()),
])

# Parsear el JSON
df_parsed = df_raw.select(
    F.from_json(F.col("value").cast("string"), schema).alias("data")
).select("data.*")

print("Esquema del DataFrame parseado:")
df_parsed.printSchema()

## 4. Window Aggregation con Watermark

Ahora aplicamos una **ventana temporal** para agregar datos. Calculamos el total de ventas y numero de transacciones por producto en ventanas de 1 minuto.

El **watermark** de 30 segundos indica que esperamos hasta 30 segundos por datos retrasados.

In [ ]:
df_windowed = df_parsed \
    .withColumn("event_time", F.to_timestamp("timestamp")) \
    .withWatermark("event_time", "30 seconds") \
    .groupBy(
        F.window("event_time", "1 minute"),
        "producto"
    ).agg(
        F.sum("monto").alias("total_ventas"),
        F.count("*").alias("num_transacciones")
    )

print("Pipeline de ventana temporal definido.")
print("Agrupando por: ventana de 1 minuto + producto")
print("Metricas: total_ventas, num_transacciones")

## 5. Escribir a Consola (Debugging)

Para depuracion, escribimos los resultados a consola. Usamos `outputMode("complete")` porque estamos haciendo agregaciones.

In [ ]:
query = df_windowed.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .trigger(processingTime="10 seconds") \
    .start()

print("Query de streaming iniciada. Esperando 30 segundos...")
time.sleep(30)  # Dejar correr 30 segundos
query.stop()
print("Query detenida.")

## 6. Escribir a Parquet

En produccion, es comun escribir los resultados de streaming a archivos **Parquet** para analisis posterior. Usamos `outputMode("append")` para ir acumulando resultados.

El **checkpointLocation** permite a Spark recuperarse de fallos sin perder el estado.

In [ ]:
query_parquet = df_parsed.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", "/home/jovyan/datos/streaming/output/transacciones") \
    .option("checkpointLocation", "/tmp/checkpoint_transacciones") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Escribiendo stream a Parquet...")
time.sleep(30)  # Dejar correr 30 segundos
query_parquet.stop()
print("Escritura a Parquet detenida.")

In [ ]:
# Verificar los datos escritos en Parquet
df_verificacion = spark.read.parquet("/home/jovyan/datos/streaming/output/transacciones")
print(f"Registros escritos en Parquet: {df_verificacion.count()}")
df_verificacion.show(5)

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

**Nota:** Antes de cada ejercicio, asegurate de que el generador de datos siga activo. Si ya termino, ejecuta nuevamente la celda del paso 1.

In [ ]:
# Reiniciar generador si es necesario
proceso_generador = subprocess.Popen([
    "python", "/home/jovyan/scripts/generar_datos_streaming.py",
    "--tipo", "transacciones", "--velocidad", "5", "--duracion", "120",
    "--topic", "bigdata-transacciones"
])
print("Generador reiniciado (5 eventos/seg por 2 min)")

In [ ]:
# =============================================================
# EJERCICIO 1: Contar transacciones por tienda en ventanas
# =============================================================
# Pipeline: ventana de 1 minuto agrupado por tienda_id

df_tiendas = df_parsed     .withColumn("event_time", F.to_timestamp("timestamp"))     .withWatermark("event_time", "30 seconds")     .groupBy(
        F.window("event_time", "1 minute"),
        "tienda_id"
    )     .agg(F.count("*").alias("num_transacciones"))

# Escribir a consola
query = df_tiendas.writeStream     .outputMode("complete")     .format("console")     .option("truncate", False)     .trigger(processingTime="10 seconds")     .start()

print("Query iniciada: conteo por tienda en ventanas de 1 minuto")
time.sleep(30)
query.stop()
print("Query detenida.")

> **Nota docente:** Este ejercicio combina tres conceptos clave: (1) conversion de string a timestamp con `to_timestamp()`, (2) watermark para manejar datos tardios, y (3) window aggregation con `groupBy(window(...), col)`. El `outputMode("complete")` es necesario porque estamos haciendo agregaciones; `append` no funcionaria aqui porque los grupos pueden cambiar al llegar nuevos datos. Error comun: olvidar el `withWatermark` o ponerlo despues del `groupBy` (debe ir antes). El `trigger(processingTime="10 seconds")` indica que Spark procesara micro-batches cada 10 segundos.

In [ ]:
# =============================================================
# EJERCICIO 2: Monto promedio por metodo de pago
# =============================================================
# Pipeline: ventana de 2 minutos agrupado por metodo_pago

df_metodo = df_parsed     .withColumn("event_time", F.to_timestamp("timestamp"))     .withWatermark("event_time", "1 minute")     .groupBy(
        F.window("event_time", "2 minutes"),
        "metodo_pago"
    )     .agg(
        F.avg("monto").alias("monto_promedio"),
        F.sum("monto").alias("monto_total"),
        F.count("*").alias("num_transacciones")
    )

# Escribir a consola
query_metodo = df_metodo.writeStream     .outputMode("complete")     .format("console")     .option("truncate", False)     .trigger(processingTime="10 seconds")     .start()

print("Query iniciada: monto promedio por metodo de pago en ventanas de 2 min")
time.sleep(30)
query_metodo.stop()
print("Query detenida.")

> **Nota docente:** La diferencia con el ejercicio anterior es la ventana mas grande (2 minutos) y el watermark mas amplio (1 minuto). Esto significa que Spark esperara hasta 1 minuto por datos retrasados antes de cerrar una ventana. Se calculan tres metricas simultaneamente con `.agg()`, demostrando que se pueden combinar multiples funciones de agregacion. Punto de discusion: como afecta el tamano de la ventana a la latencia vs. precision de los resultados.

---
## Resumen

En esta actividad aprendimos:

1. **Structured Streaming:** Spark trata streams como tablas ilimitadas procesadas en micro-batches
2. **readStream desde Kafka:** Lectura continua de topics con `spark.readStream.format("kafka")`
3. **Parseo de JSON:** Uso de `from_json()` con esquema para extraer campos del mensaje
4. **Watermarks:** Mecanismo para manejar datos retrasados y liberar estado
5. **Window Aggregations:** Agrupacion por ventanas de tiempo para calcular metricas
6. **Output Modes:** append, complete y update segun el tipo de operacion
7. **writeStream:** Escritura a consola, Parquet y otros destinos

---
## Desafio Extra (Opcional)

**Deteccion de anomalias en streaming:**

Crea un pipeline que filtre transacciones con monto mayor a 1,000,000 (posible fraude) y escriba las alertas a un topic de Kafka separado llamado "alertas-fraude".

In [ ]:
# =============================================================
# DESAFIO: Deteccion de anomalias en streaming
# =============================================================
# Pipeline que filtra posibles fraudes y los envia a un topic de Kafka

# Filtrar transacciones sospechosas (monto > 1,000,000)
df_alertas = df_parsed     .filter(F.col("monto") > 1000000)     .withColumn("alerta", F.lit("POSIBLE FRAUDE"))     .withColumn("detectado_en", F.current_timestamp())

# Convertir a JSON y escribir al topic de Kafka "alertas-fraude"
query_alertas = df_alertas     .select(F.to_json(F.struct("*")).alias("value"))     .writeStream     .outputMode("append")     .format("kafka")     .option("kafka.bootstrap.servers", "kafka:29092")     .option("topic", "alertas-fraude")     .option("checkpointLocation", "/tmp/checkpoint_alertas")     .trigger(processingTime="10 seconds")     .start()

print("Pipeline de deteccion de fraude iniciado.")
print("Filtrando transacciones con monto > $1,000,000...")
time.sleep(30)
query_alertas.stop()
print("Pipeline detenido.")

> **Nota docente:** Este desafio introduce un patron de produccion real: deteccion de anomalias en streaming con re-publicacion a otro topic de Kafka. Conceptos clave: (1) `F.lit()` para agregar columnas constantes, (2) `F.current_timestamp()` para registrar cuando se detecto la alerta, (3) `to_json(struct("*"))` para serializar todo el registro como JSON, (4) `outputMode("append")` es obligatorio cuando se escribe a Kafka (no soporta `complete`). En produccion, el topic "alertas-fraude" seria consumido por un sistema de notificaciones que alerte al equipo de seguridad.

In [ ]:
# Detener todas las queries activas y la SparkSession
for q in spark.streams.active:
    q.stop()
    print(f"Query '{q.name}' detenida.")

# Detener el generador de datos
try:
    proceso_generador.terminate()
    print("Generador de datos detenido.")
except:
    print("Generador ya habia terminado.")

spark.stop()
print("SparkSession detenida correctamente.")